In [ ]:
!pip install gymnasium[atari]
!pip install autorom
!AutoROM --accept-license

In [ ]:
import torch
import torch.nn as nn
import wandb
import gymnasium as gym
from gymnasium.vector import SyncVectorEnv
from gymnasium.wrappers import RecordVideo, FrameStackObservation, AtariPreprocessing, RecordVideo
import ale_py
import numpy as np
import time

In [ ]:
GAME_ID = "ALE/Riverraid-v5"
MAX_TRAIN_STEPS = 5_000_000
ROLLOUT_STEP = 256
EVAL_STEPS = MAX_TRAIN_STEPS // 500
MAX_EVAL_STEP = 27000
FRAME_STACK_SIZE = 4
NUM_ENVS = 16
EPOCHS = 5
EVAL_LOOP = 5
LR = 1.5e-4
GLOBAL_STEPS = 0
BATCH_SIZE = 512
REWARD_DECAY = 0.99
GAE_LAMBDA = 0.95
PPO_CLIP = 0.15
INITIAL_ENTROPY_BETA = 0.01
VALUE_LOSS_BETA = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def make_env(id):
  def _init():
      env = gym.make(id, frameskip=1)
      env = AtariPreprocessing(env)
      env = FrameStackObservation(env, 4)

      return env
  return _init

In [ ]:
def wandb_runs():
  wandb.login(key=" ")
  run = wandb.init(
    entity=" ",
    project="vectorized-env",
    name = "vectorized-PPO"
    )
  return run

In [ ]:
env = SyncVectorEnv([make_env(GAME_ID) for _ in range(NUM_ENVS)])

In [ ]:
def evaluation(actornet, rec_video = False):

  eval_env = gym.make(GAME_ID, frameskip=1, max_episode_steps= MAX_EVAL_STEP, render_mode = 'rgb_array')
  eval_env = AtariPreprocessing(eval_env)

  if rec_video:
    video_dir = f"videos/{int(time.time())}"
    eval_env = RecordVideo(eval_env, video_folder=video_dir, episode_trigger=lambda ep: True)

  eval_env = FrameStackObservation(eval_env, 4)

  total_reward = 0
  total_step = 0

  with torch.no_grad():

    for _ in range(EVAL_LOOP):

      state = eval_env.reset()[0]
      done = False

      while not done:
        state = torch.tensor(state, device=DEVICE).float().unsqueeze(0)
        action = actornet(state).argmax().item()
        next_state, reward, terminated, truncation, _ = eval_env.step(action)
        done = terminated or truncation
        state = next_state
        total_reward += float(reward)
        total_step += 1

  average_reward = total_reward / EVAL_LOOP
  average_step = total_step / EVAL_LOOP
  eval_env.close()

  return average_reward, average_step

In [ ]:
state = env.reset()[0]

In [ ]:
evaluation(actornetwork, True)

In [ ]:
class ActorNet(nn.Module):

  def __init__(self):

      super().__init__()
      self.convolution = nn.Sequential(
          nn.Conv2d(4, 32, kernel_size=8, stride=4),
          nn.ReLU(),
          nn.Conv2d(32, 64, kernel_size=4, stride=2),
          nn.ReLU(),
          nn.Conv2d(64, 64, kernel_size=3, stride=1),
          nn.ReLU()
      )
      self.linear = nn.Sequential(
          nn.Linear(3136, 512),
          nn.ReLU(),
          nn.Linear(512, 512),
          nn.ReLU(),
          nn.Linear(512, env.single_action_space.n)
      )

  def forward(self, x):
      x = x/255.
      x = self.convolution(x)
      x = x.view(x.size(0), -1)
      return self.linear(x)

  def get_action(self, x, actions = None):

      logits = self.forward(x)
      dist = torch.distributions.Categorical(logits=logits)
      action = dist.sample() if actions is None else actions
      log_prob = dist.log_prob(action)
      entropy = dist.entropy()
      return action, log_prob, entropy

In [ ]:
class CriticNet(nn.Module):

  def __init__(self):

    super().__init__()
    self.convolution = nn.Sequential(
        nn.Conv2d(4, 32, kernel_size=8, stride=4),
        nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=4, stride=2),
        nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, stride=1),
        nn.ReLU()
    )
    self.linear = nn.Sequential(
        nn.Linear(3136, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 1)
    )

  def forward(self, x):
    x = x/255.
    x = self.convolution(x)
    x = x.view(x.size(0), -1)
    return self.linear(x)

In [ ]:
actornetwork = ActorNet().to(DEVICE)
criticnetwork = CriticNet().to(DEVICE)

In [ ]:
print(f'actor--network: {sum(p.numel() for p in actornetwork.parameters()) / 1e6 :.3f} M \ncritic-network: {sum(p.numel() for p in criticnetwork.parameters())/1e6:.3f} M')

print(f"\n{DEVICE} is being used")

In [ ]:
optimizer = torch.optim.Adam([
    {"params": actornetwork.parameters(), "lr": LR},
    {"params": criticnetwork.parameters(), "lr": LR}
    ])

In [ ]:
state, _ = env.reset()  # (5, 4, 84, 84)  --> (nEnv, frameStack, H, W)

runs = wandb_runs()

In [ ]:
for step in range(MAX_TRAIN_STEPS):

  all_states = []
  all_actions = []
  all_rewards = []
  all_next_states = []
  all_log_probs = []
  all_dones = []
  all_terminated = []

  done = False

  current_rollout = 0

  training_reward = torch.zeros((NUM_ENVS), device = DEVICE)

  while current_rollout < ROLLOUT_STEP:

    stateTensor = torch.tensor(state, device=DEVICE).float()

    with torch.no_grad():
      action, log_prob, _ = actornetwork.get_action(stateTensor, None)
      next_state, reward, terminated, truncated, _ = env.step(action.cpu().numpy())  # type = ignore

    # print(reward)
    done = terminated | truncated

    training_reward += torch.tensor(reward, device=DEVICE)

    reward = torch.tensor(reward, dtype=torch.float).to(DEVICE)

    all_states.append(stateTensor)
    all_actions.append(action)
    all_rewards.append(reward)
    all_log_probs.append(log_prob)
    all_next_states.append(torch.tensor(next_state, dtype=torch.float).to(DEVICE))
    all_dones.append(torch.tensor(done, dtype=torch.long).to(DEVICE))
    all_terminated.append(torch.tensor(terminated, dtype=torch.long).to(DEVICE))

    state = next_state


    if GLOBAL_STEPS % EVAL_STEPS == 0:
      avg_eval_reward, avg_step = evaluation(actornetwork)
      runs.log({
          "average-eval-return": avg_eval_reward,
          "average-eval-steps": avg_step,
      }, GLOBAL_STEPS)


    current_rollout += 1
    GLOBAL_STEPS += 1

  all_states = torch.stack(all_states)
  all_actions = torch.stack(all_actions)                          # [ ROLLOUTS, i_Env ]
  all_next_states = torch.stack(all_next_states)
  all_rewards = torch.stack(all_rewards)
  all_dones = torch.stack(all_dones)
  all_log_probs = torch.stack(all_log_probs)
  all_terminated = torch.stack(all_terminated)

  # ================ general advantage estimation ================= #

  delta_error = torch.zeros_like(all_rewards).to(DEVICE)
  GAE = torch.zeros_like(all_rewards).to(DEVICE)
  all_returns = torch.zeros_like(all_rewards).to(DEVICE)

  with torch.no_grad():

    for i_env in range(NUM_ENVS):

      V_st = criticnetwork(all_states[: , i_env])
      V_st1 = criticnetwork(all_next_states[: , i_env])

      delta_error[: , i_env] = all_rewards[: , i_env] + REWARD_DECAY * V_st1.view(-1) * (1 - all_terminated[:, i_env]) - V_st.view(-1)

      reversed_delta_error = reversed(delta_error[:, i_env])
      GAE_t = 0
      idx = ROLLOUT_STEP - 1

      for delta in reversed_delta_error:
          done = all_dones[idx, i_env]
          GAE_t = delta + REWARD_DECAY*GAE_LAMBDA*GAE_t*(1 - done)
          GAE[idx, i_env] = GAE_t
          idx-= 1

      all_returns[:, i_env] = GAE[:, i_env] + V_st.view(-1)

  # ================= flattening everything ==================== #

  all_states = all_states.view(-1, FRAME_STACK_SIZE, 84, 84)
  all_actions = all_actions.view(-1)
  all_rewards = all_rewards.view(-1)
  all_next_states = all_next_states.view(-1, FRAME_STACK_SIZE, 84, 84)
  all_log_probs = all_log_probs.view(-1)
  all_dones = all_dones.view(-1)
  all_terminated = all_terminated.view(-1)
  all_returns = all_returns.view(-1)
  GAE = GAE.view(-1)

  GAE = (GAE - GAE.mean()) /(GAE.std() + 1e-8)

  # ================ ppo update loop ============================ #

  entropy_loss = 0
  value_loss = 0
  policy_loss = 0
  kl_divergence = 0
  step = 0

  ENTROPY_BETA = INITIAL_ENTROPY_BETA*(1 - (GLOBAL_STEPS/MAX_TRAIN_STEPS))

  for epoch in range(EPOCHS):

    idxes = torch.randperm(all_states.size(0))

    for idx in range(0, idxes.size(0), BATCH_SIZE):

        batch_idxes = idxes[idx:idx+BATCH_SIZE]
        batch_states = all_states[batch_idxes]
        batch_actions = all_actions[batch_idxes]
        batch_next_states = all_next_states[batch_idxes]
        old_log_probs = all_log_probs[batch_idxes]
        batch_GAE = GAE[batch_idxes]
        batch_returns = all_returns[batch_idxes]

        _, new_log_probs, mb_entropy = actornetwork.get_action(batch_states, batch_actions)

        ratio = torch.exp(new_log_probs - old_log_probs)

        ppo_expected = torch.min(ratio * batch_GAE, torch.clamp(ratio, 1 - PPO_CLIP, 1 + PPO_CLIP) * batch_GAE).mean()

        criticLoss = torch.nn.functional.mse_loss(criticnetwork(batch_states).view(-1), batch_returns)

        entropy = mb_entropy.mean()

        ppo_loss = -ppo_expected - ENTROPY_BETA * entropy + VALUE_LOSS_BETA * criticLoss

        kl_div = (old_log_probs - new_log_probs).mean()

        optimizer.zero_grad()
        ppo_loss.backward()
        torch.nn.utils.clip_grad_norm_(actornetwork.parameters(), 0.5)
        torch.nn.utils.clip_grad_norm_(criticnetwork.parameters(), 0.5)
        optimizer.step()

        policy_loss += ppo_loss.item()
        value_loss += criticLoss.item()
        entropy_loss += entropy.item()

        kl_divergence += kl_div.item()
        step += 1


  runs.log({
      "actor-loss": policy_loss/step,
      "critic-loss": value_loss/step,
      "entropy-loss": entropy_loss/step,
      "kl-divergence": kl_divergence/step,
      "training-reward": training_reward.mean().item(),
      "global-steps": GLOBAL_STEPS,
      "advantage-std": GAE.std().item(),
      "advantage-max": GAE.max().item(),
      "delta mean": delta_error.mean().item(),
      "V_st mean": V_st.mean().item()
  }, GLOBAL_STEPS)